# Лабораторная работа: Многослойный перцептрон (MLP)

**Задача:** мультиклассификация планетарных архетипов экзопланет на основе звёздной фотометрии и орбитальных параметров.

**Датасет:** `nasa_exoplanets_archive.csv` — архив подтверждённых экзопланет NASA. После фильтрации пропусков и ограничения до 1024 объектов.

**Фреймворк:** PyTorch

## Признаки, по которым обучается модель

| Признак | Что означает |
|---|---|
| `st_teff` | Эффективная температура звезды-хоста (K) |
| `st_mass` | Масса звезды-хоста (в массах Солнца) |
| `st_met` | Металличность звезды-хоста (dex) |
| `pl_orbper` | Орбитальный период планеты (дни) |

## Целевая переменная — `planetary_archetype` (5 классов)

| Класс | Описание |
|---|---|
| `terrestrial_rocky` | Каменистая планета земного типа |
| `super_earth` | Суперземля |
| `sub_neptune_mini_gas` | Мини-нептун / суб-нептун |
| `gas_giant_jupiter` | Газовый гигант юпитерианского типа |
| `super_jupiter_extreme` | Экстремальный суперюпитер |

---
## 0. Импорт библиотек и константы

In [117]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score
)
from typing import List

# ── Константы ──
DATA_PATH = 'nasa_exoplanets_archive.csv'
TARGET_COLUMN = 'planetary_archetype'
FEATURE_COLUMNS = ['st_teff', 'st_mass', 'st_met', 'pl_orbper']
MAX_SAMPLES = 1024

SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.25
BATCH_SIZE = 32

INPUT_DIM = len(FEATURE_COLUMNS)
NUM_CLASSES = 5
HIDDEN_SIZES = [64, 32]
EPOCHS = 100
LR = 1e-3
DROPOUT_RATE = 0.3

ARCHITECTURES = {
    'Shallow-Narrow [16]':          [16],
    'Shallow-Wide [128]':           [128],
    'Medium [64, 32]':              [64, 32],
    'Deep-Narrow [32, 16, 8]':      [32, 16, 8],
    'Deep-Wide [128, 64, 32]':      [128, 64, 32],
    'Very Deep [64, 32, 16, 8, 4]': [64, 32, 16, 8, 4],
}

OVERFIT_GAP_THRESHOLD = 0.05
PLOT_TEMPLATE = 'plotly_white'
CLASS_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

np.random.seed(SEED)
torch.manual_seed(SEED)

---
## 1. Загрузка и первичный анализ данных

In [118]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Исходный размер: {df_raw.shape}')

cols_needed = FEATURE_COLUMNS + [TARGET_COLUMN]
df = df_raw[cols_needed].dropna().reset_index(drop=True)
print(f'После удаления пропусков: {df.shape}')

if len(df) > MAX_SAMPLES:
    df, _ = train_test_split(df, train_size=MAX_SAMPLES, random_state=SEED, stratify=df[TARGET_COLUMN])
    df = df.reset_index(drop=True)
    print(f'После ограничения до {MAX_SAMPLES}: {df.shape}')

df.head()

Исходный размер: (6445, 25)
После удаления пропусков: (4368, 5)
После ограничения до 1024: (1024, 5)


,st_teff,st_mass,st_met,pl_orbper,planetary_archetype
0,5400.0,0.920,0.13,1684.000000,gas_giant_jupiter
1,5772.0,0.999,0.00,2.762660,terrestrial_rocky
2,6322.0,1.250,0.11,4.919584,sub_neptune_mini_gas
3,3810.0,0.580,-0.10,20.523000,super_earth
4,5500.0,0.930,0.05,4.339465,super_earth


In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1024 entries, 0 to 1023
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   st_teff              1024 non-null   float64
 1   st_mass              1024 non-null   float64
 2   st_met               1024 non-null   float64
 3   pl_orbper            1024 non-null   float64
 4   planetary_archetype  1024 non-null   object 
dtypes: float64(4), object(1)
memory usage: 40.1+ KB


In [120]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
st_teff,1024.0,5335.102129,850.346595,2953.000000,4928.750000,5549.50000,5933.000000,7930.0000
st_mass,1024.0,0.970253,0.410097,0.120000,0.802000,0.96000,1.112000,8.7600
st_met,1024.0,0.009533,0.190660,-0.950000,-0.080000,0.02000,0.130000,0.5450
pl_orbper,1024.0,204.211568,844.813931,0.224196,4.113582,9.87658,38.688575,11611.2975


In [121]:
print('Пропущенные значения:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Пропуски отсутствуют')

Пропущенные значения:
Пропуски отсутствуют


In [122]:
print('Распределение целевой переменной:')
print(df[TARGET_COLUMN].value_counts())
print('\nДоля классов:')
print(df[TARGET_COLUMN].value_counts(normalize=True).round(3))

Распределение целевой переменной:
planetary_archetype
sub_neptune_mini_gas     330
gas_giant_jupiter        321
super_earth              221
terrestrial_rocky         96
super_jupiter_extreme     56
Name: count, dtype: int64

Доля классов:
planetary_archetype
sub_neptune_mini_gas     0.322
gas_giant_jupiter        0.313
super_earth              0.216
terrestrial_rocky        0.094
super_jupiter_extreme    0.055
Name: proportion, dtype: float64


---
## 2. Визуализация данных

### Столбчатая диаграмма классов

In [123]:
class_counts = df[TARGET_COLUMN].value_counts()
fig = go.Figure(go.Bar(
    x=class_counts.index, y=class_counts.values,
    marker_color=CLASS_COLORS[:len(class_counts)],
    text=class_counts.values, textposition='auto'
))
fig.update_layout(
    title='Распределение планетарных архетипов',
    xaxis_title='Архетип', yaxis_title='Количество',
    template=PLOT_TEMPLATE, width=900, height=450
)
fig.show()

### Гистограммы признаков по классам
Что показывает: распределения признаков отдельно для каждого архетипа. Чем меньше перекрытие, тем полезнее признак.

In [124]:
classes = df[TARGET_COLUMN].unique()
fig = make_subplots(rows=1, cols=len(FEATURE_COLUMNS), subplot_titles=FEATURE_COLUMNS)
for i, col in enumerate(FEATURE_COLUMNS):
    for j, cls in enumerate(classes):
        subset = df[df[TARGET_COLUMN] == cls]
        fig.add_trace(go.Histogram(
            x=subset[col], name=cls, marker_color=CLASS_COLORS[j % len(CLASS_COLORS)],
            opacity=0.5, nbinsx=20, showlegend=(i == 0)
        ), row=1, col=i + 1)
    fig.update_xaxes(title_text=col, row=1, col=i + 1)

fig.update_layout(
    title_text='Гистограммы признаков по классам',
    barmode='overlay', template=PLOT_TEMPLATE,
    height=500, width=1200
)
fig.show()

### Boxplot
Каждый ящик показывает медиану, разброс и выбросы признака для каждого класса.

In [125]:
fig = make_subplots(rows=1, cols=len(FEATURE_COLUMNS), subplot_titles=FEATURE_COLUMNS)
for i, col in enumerate(FEATURE_COLUMNS):
    for j, cls in enumerate(classes):
        subset = df[df[TARGET_COLUMN] == cls]
        fig.add_trace(go.Box(
            y=subset[col], name=cls, marker_color=CLASS_COLORS[j % len(CLASS_COLORS)],
            showlegend=(i == 0)
        ), row=1, col=i + 1)

fig.update_layout(
    title_text='Boxplot признаков по классам',
    template=PLOT_TEMPLATE, height=500, width=1200
)
fig.show()

### Корреляция

In [126]:
corr = df[FEATURE_COLUMNS].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values, x=FEATURE_COLUMNS, y=FEATURE_COLUMNS,
    colorscale='RdBu', zmin=-1, zmax=1, colorbar=dict(title='Корреляция')
))

for row_i, y_name in enumerate(FEATURE_COLUMNS):
    for col_i, x_name in enumerate(FEATURE_COLUMNS):
        value = corr.iloc[row_i, col_i]
        fig.add_annotation(
            x=x_name, y=y_name, text=f'{value:.2f}', showarrow=False,
            font=dict(color='white' if abs(value) > 0.5 else 'black', size=12)
        )

fig.update_layout(
    title='Корреляционная матрица признаков',
    template=PLOT_TEMPLATE, width=600, height=500,
    yaxis=dict(autorange='reversed')
)
fig.show()

### PCA-проекция

In [127]:
scaler_pca = StandardScaler()
X_pca = scaler_pca.fit_transform(df[FEATURE_COLUMNS])
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_pca)

le_temp = LabelEncoder()
labels = le_temp.fit_transform(df[TARGET_COLUMN])

fig = go.Figure()
for j, cls in enumerate(le_temp.classes_):
    mask = labels == j
    fig.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1],
        mode='markers', name=cls,
        marker=dict(color=CLASS_COLORS[j % len(CLASS_COLORS)], size=6, opacity=0.7)
    ))

fig.update_layout(
    title='PCA-проекция данных',
    xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
    yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    template=PLOT_TEMPLATE, width=800, height=600
)
fig.show()

---
## 3. Предобработка и разделение данных

Разбиение: **train / validation / test = 60% / 20% / 20%**

- **train** — для обучения модели (подгонка весов);
- **validation** — для подбора гиперпараметров и контроля переобучения;
- **test** — финальная, независимая оценка качества модели.

In [128]:
X = df[FEATURE_COLUMNS].values
le = LabelEncoder()
y = le.fit_transform(df[TARGET_COLUMN])
print(f'Классы: {list(le.classes_)}')

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=VAL_SIZE, random_state=SEED, stratify=y_train_val
)

print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')
for i, cls in enumerate(le.classes_):
    print(f'  {cls}: train={np.mean(y_train==i):.3f}, val={np.mean(y_val==i):.3f}, test={np.mean(y_test==i):.3f}')

Классы: ['gas_giant_jupiter', 'sub_neptune_mini_gas', 'super_earth', 'super_jupiter_extreme', 'terrestrial_rocky']
Train: 614, Val: 205, Test: 205
  gas_giant_jupiter: train=0.313, val=0.312, test=0.317
  sub_neptune_mini_gas: train=0.322, val=0.322, test=0.322
  super_earth: train=0.215, val=0.220, test=0.215
  super_jupiter_extreme: train=0.055, val=0.054, test=0.054
  terrestrial_rocky: train=0.094, val=0.093, test=0.093


In [129]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

def to_tensors(X, y):
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long))

X_train_t, y_train_t = to_tensors(X_train, y_train)
X_val_t, y_val_t = to_tensors(X_val, y_val)
X_test_t, y_test_t = to_tensors(X_test, y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

---
## 4. Построение и обучение MLP

In [130]:
class MLP(nn.Module):

    def __init__(self, inputDim: int,
                 hiddenShape: List[int],
                 outputDim: int = NUM_CLASSES,
                 fActivation=nn.ReLU,
                 dropoutRate: float = DROPOUT_RATE):
        super().__init__()

        self.inputModule = nn.Sequential(
            nn.Linear(inputDim, hiddenShape[0]),
            nn.BatchNorm1d(hiddenShape[0]),
            fActivation(),
            nn.Dropout(dropoutRate),
        )

        self.hiddenModule = nn.Sequential(
            *[nn.Sequential(
                nn.Linear(hiddenShape[i], hiddenShape[i + 1]),
                nn.BatchNorm1d(hiddenShape[i + 1]),
                fActivation(),
                nn.Dropout(dropoutRate),
              ) for i in range(len(hiddenShape) - 1)]
        )

        self.outputModule = nn.Linear(hiddenShape[-1], outputDim)

    def forward(self, x):
        x = self.inputModule(x)
        x = self.hiddenModule(x)
        x = self.outputModule(x)
        return x

model = MLP(INPUT_DIM, HIDDEN_SIZES)
print(model)

MLP(
  (inputModule): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (hiddenModule): Sequential(
    (0): Sequential(
      (0): Linear(in_features=64, out_features=32, bias=True)
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
    )
  )
  (outputModule): Linear(in_features=32, out_features=5, bias=True)
)


In [131]:
def evaluate(model, loader, criterion):
    loss_sum, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        logits = model(xb)
        loss_sum += criterion(logits, yb).item() * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return loss_sum / total, correct / total


def train_model(model, train_loader, val_loader, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            t_loss, t_acc = evaluate(model, train_loader, criterion)
            v_loss, v_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

    return history

In [132]:
history = train_model(model, train_loader, val_loader, EPOCHS, LR)

### 4.1 Визуализация обучения и проверка переобучения

In [133]:
def plot_history(history, title=''):
    epochs = list(range(1, len(history['train_loss']) + 1))
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[f'Функция потерь {title}', f'Точность {title}'])

    fig.add_trace(go.Scatter(x=epochs, y=history['train_loss'], name='Train Loss',
                             line=dict(color='#1f77b4')), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=history['val_loss'], name='Val Loss',
                             line=dict(color='#ff7f0e')), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=history['train_acc'], name='Train Acc',
                             line=dict(color='#1f77b4')), row=1, col=2)
    fig.add_trace(go.Scatter(x=epochs, y=history['val_acc'], name='Val Acc',
                             line=dict(color='#ff7f0e')), row=1, col=2)

    fig.update_xaxes(title_text='Эпоха')
    fig.update_yaxes(title_text='Loss', row=1, col=1)
    fig.update_yaxes(title_text='Accuracy', row=1, col=2)
    fig.update_layout(height=500, width=1000, template=PLOT_TEMPLATE)
    fig.show()

    gap = history['train_acc'][-1] - history['val_acc'][-1]
    print(f'Train Acc: {history["train_acc"][-1]:.4f}')
    print(f'Val Acc:   {history["val_acc"][-1]:.4f}')
    print(f'Разрыв (train - val): {gap:.4f}')
    if gap > OVERFIT_GAP_THRESHOLD:
        print('⚠ Наблюдается переобучение (разрыв > 5%)')
    else:
        print('✓ Переобучение не выражено')

plot_history(history, '(базовая модель: [64, 32])')

Train Acc: 0.5326
Val Acc:   0.5707
Разрыв (train - val): -0.0382
✓ Переобучение не выражено


### 4.2 Оценка на тестовой выборке

In [134]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            logits = model(xb)
            probs = torch.softmax(logits, dim=1).numpy()
            preds = logits.argmax(dim=1).numpy()
            all_probs.append(probs)
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())
    return np.array(all_labels), np.array(all_preds), np.vstack(all_probs)

y_true, y_pred, y_prob = evaluate_model(model, test_loader)

print('=== Результаты на тестовой выборке ===')
print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision (macro): {precision_score(y_true, y_pred, average="macro"):.4f}')
print(f'Recall (macro):    {recall_score(y_true, y_pred, average="macro"):.4f}')
print(f'F1 (macro):        {f1_score(y_true, y_pred, average="macro"):.4f}')
print(f'ROC AUC (ovr):     {roc_auc_score(y_true, y_prob, multi_class="ovr"):.4f}')
print()
print(classification_report(y_true, y_pred, target_names=le.classes_))

=== Результаты на тестовой выборке ===
Accuracy:  0.5171
Precision (macro): 0.5138
Recall (macro):    0.3748
F1 (macro):        0.3684
ROC AUC (ovr):     0.7275

                       precision    recall  f1-score   support

    gas_giant_jupiter       0.76      0.68      0.72        65
 sub_neptune_mini_gas       0.41      0.83      0.55        66
          super_earth       0.40      0.09      0.15        44
super_jupiter_extreme       1.00      0.27      0.43        11
    terrestrial_rocky       0.00      0.00      0.00        19

             accuracy                           0.52       205
            macro avg       0.51      0.37      0.37       205
         weighted avg       0.51      0.52      0.46       205



### Confusion Matrix

In [135]:
cm = confusion_matrix(y_true, y_pred)
fig = go.Figure(go.Heatmap(
    z=cm, x=list(le.classes_), y=list(le.classes_),
    colorscale='Blues', showscale=False,
    text=cm, texttemplate='%{text}', textfont=dict(size=14)
))
fig.update_layout(
    title='Confusion Matrix',
    xaxis_title='Предсказано', yaxis_title='Истинное',
    yaxis=dict(autorange='reversed'),
    template=PLOT_TEMPLATE, width=700, height=600
)
fig.show()

---
## 5. Исследование влияния глубины и ширины MLP

In [136]:
results = {}

for name, hidden in ARCHITECTURES.items():
    torch.manual_seed(SEED)
    m = MLP(INPUT_DIM, hidden)
    h = train_model(m, train_loader, val_loader, EPOCHS, LR)
    y_true_exp, y_pred_exp, y_prob_exp = evaluate_model(m, test_loader)
    acc = accuracy_score(y_true_exp, y_pred_exp)
    f1 = f1_score(y_true_exp, y_pred_exp, average='macro')
    auc = roc_auc_score(y_true_exp, y_prob_exp, multi_class='ovr')
    n_params = sum(p.numel() for p in m.parameters())
    results[name] = {
        'hidden': hidden, 'params': n_params,
        'test_acc': acc, 'test_f1': f1, 'test_auc': auc, 'history': h
    }
    print(f'{name:35s} | params={n_params:>6,} | acc={acc:.4f} | f1={f1:.4f} | auc={auc:.4f}')

Shallow-Narrow [16]                 | params=   197 | acc=0.4878 | f1=0.3626 | auc=0.7155
Shallow-Wide [128]                  | params= 1,541 | acc=0.4878 | f1=0.3586 | auc=0.7147
Medium [64, 32]                     | params= 2,757 | acc=0.5171 | f1=0.3684 | auc=0.7275
Deep-Narrow [32, 16, 8]             | params=   981 | acc=0.4927 | f1=0.2758 | auc=0.7151
Deep-Wide [128, 64, 32]             | params=11,589 | acc=0.5073 | f1=0.3632 | auc=0.7166
Very Deep [64, 32, 16, 8, 4]        | params= 3,373 | acc=0.4780 | f1=0.2468 | auc=0.7230


In [137]:
res_df = pd.DataFrame({
    k: {'Параметры': v['params'], 'Accuracy': v['test_acc'],
         'F1': v['test_f1'], 'AUC': v['test_auc']}
    for k, v in results.items()
}).T
res_df = res_df.sort_values('Accuracy', ascending=False)
print(res_df.to_string())

                              Параметры  Accuracy        F1       AUC
Medium [64, 32]                  2757.0  0.517073  0.368433  0.727450
Deep-Wide [128, 64, 32]         11589.0  0.507317  0.363203  0.716640
Deep-Narrow [32, 16, 8]           981.0  0.492683  0.275777  0.715102
Shallow-Narrow [16]               197.0  0.487805  0.362570  0.715531
Shallow-Wide [128]               1541.0  0.487805  0.358606  0.714680
Very Deep [64, 32, 16, 8, 4]     3373.0  0.478049  0.246760  0.722965


### Кривые обучения по архитектурам

In [138]:
names_list = list(results.keys())
fig = make_subplots(rows=2, cols=3, subplot_titles=[
    f'{name}<br>acc={results[name]["test_acc"]:.3f}' for name in names_list
])
for i, name in enumerate(names_list):
    row, col = i // 3 + 1, i % 3 + 1
    h = results[name]['history']
    epochs = list(range(1, len(h['train_loss']) + 1))
    fig.add_trace(go.Scatter(x=epochs, y=h['train_loss'], name='Train',
                             showlegend=(i == 0)), row=row, col=col)
    fig.add_trace(go.Scatter(x=epochs, y=h['val_loss'], name='Validation',
                             showlegend=(i == 0)), row=row, col=col)
fig.update_layout(title='Сравнение архитектур: кривые обучения',
                  template=PLOT_TEMPLATE, height=700, width=1200)
fig.show()

### Сравнение метрик по архитектурам

In [139]:
names = list(results.keys())
fig = go.Figure()
fig.add_trace(go.Bar(x=names, y=[results[n]['test_acc'] for n in names],
                     name='Accuracy', marker_color='#3498db'))
fig.add_trace(go.Bar(x=names, y=[results[n]['test_f1'] for n in names],
                     name='F1 (macro)', marker_color='#e67e22'))
fig.update_layout(title='Сравнение архитектур MLP', yaxis_title='Метрика',
                  barmode='group', template=PLOT_TEMPLATE,
                  height=550, width=1000)
fig.show()

---
## 6. Извлечение градиентов при инференсе

In [140]:
def extract_gradients(model, x_sample):
    model.eval()
    x = x_sample.clone().detach().unsqueeze(0)
    x.requires_grad_(True)

    activations = {}
    hooks = []

    def make_hook(name):
        def hook_fn(module, input, output):
            output.retain_grad()
            activations[name] = output
        return hook_fn

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            h = module.register_forward_hook(make_hook(name))
            hooks.append(h)

    output = model(x)
    output.sum().backward()

    grads = {}
    for name, act in activations.items():
        if act.grad is not None:
            grads[name] = act.grad.detach().numpy().flatten()

    for h in hooks:
        h.remove()

    return grads, activations

In [141]:
sample_idx = 0
sample_x = X_test_t[sample_idx]
sample_label = y_test[sample_idx]
print(f'Выбран объект #{sample_idx}, истинный класс: {le.classes_[sample_label]}')

grads, acts = extract_gradients(model, sample_x)

for layer_name, grad in grads.items():
    print(f'\nСлой: {layer_name}')
    print(f'  Размерность:       {grad.shape}')
    print(f'  Мин. градиент:     {grad.min():.6f}')
    print(f'  Макс. градиент:    {grad.max():.6f}')
    print(f'  Средний |градиент|: {np.abs(grad).mean():.6f}')

Выбран объект #0, истинный класс: gas_giant_jupiter

Слой: inputModule.0
  Размерность:       (64,)
  Мин. градиент:     -0.500421
  Макс. градиент:    0.625684
  Средний |градиент|: 0.093389

Слой: hiddenModule.0.0
  Размерность:       (32,)
  Мин. градиент:     -0.896242
  Макс. градиент:    0.862946
  Средний |градиент|: 0.191292

Слой: outputModule
  Размерность:       (5,)
  Мин. градиент:     1.000000
  Макс. градиент:    1.000000
  Средний |градиент|: 1.000000


### Визуализация градиентов

In [142]:
n_layers = len(grads)
fig = make_subplots(rows=1, cols=n_layers,
                    subplot_titles=[f'Градиенты: {name}' for name in grads])
for i, (name, grad) in enumerate(grads.items()):
    fig.add_trace(go.Bar(x=list(range(len(grad))), y=grad,
                         marker_color='steelblue', showlegend=False), row=1, col=i + 1)
    fig.update_xaxes(title_text='Нейрон', row=1, col=i + 1)
    fig.update_yaxes(title_text='Градиент', row=1, col=i + 1)
fig.update_layout(title=f'Градиенты при инференсе объекта #{sample_idx}',
                  template=PLOT_TEMPLATE, height=450, width=500 * n_layers)
fig.show()

In [143]:
layer_names = list(grads.keys())
mean_abs_grads = [np.abs(grads[name]).mean() for name in layer_names]
fig = go.Figure(go.Bar(x=layer_names, y=mean_abs_grads,
                       marker_color='#9b59b6', opacity=0.8))
fig.update_layout(title='Средний абсолютный градиент по слоям',
                  yaxis_title='Средний |градиент|', template=PLOT_TEMPLATE,
                  height=450, width=750)
fig.show()

---
## 7. Функции активации

Реализация основных функций активации на NumPy и их визуализация.

In [144]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def elu(x, alpha=1.0):
    return np.where(x > 0, x, alpha * (np.exp(x) - 1))

def swish(x):
    return x * sigmoid(x)

def softplus(x):
    return np.log(1 + np.exp(x))

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

In [145]:
activations_dict = {
    'Sigmoid': sigmoid, 'Tanh': tanh, 'ReLU': relu, 'Leaky ReLU': leaky_relu,
    'ELU': elu, 'Swish': swish, 'Softplus': softplus, 'GELU': gelu
}
x_range = np.linspace(-5, 5, 500)
colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3',
          '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']

fig = make_subplots(rows=2, cols=4, subplot_titles=list(activations_dict.keys()))
for i, ((name, func), color) in enumerate(zip(activations_dict.items(), colors)):
    row, col = i // 4 + 1, i % 4 + 1
    fig.add_trace(go.Scatter(x=x_range, y=func(x_range), mode='lines',
                             line=dict(color=color, width=2), showlegend=False), row=row, col=col)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.5, row=row, col=col)
    fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.5, row=row, col=col)
fig.update_layout(title='Функции активации', template=PLOT_TEMPLATE, height=650, width=1200)
fig.show()

In [146]:
print('Проверка значений функций активации при x = [-2, -1, 0, 1, 2]:')
test_x = np.array([-2, -1, 0, 1, 2])
for name, func in activations_dict.items():
    vals = func(test_x)
    print(f'{name:12s}: {np.round(vals, 4)}')

Проверка значений функций активации при x = [-2, -1, 0, 1, 2]:
Sigmoid     : [0.1192 0.2689 0.5    0.7311 0.8808]
Tanh        : [-0.964  -0.7616  0.      0.7616  0.964 ]
ReLU        : [0 0 0 1 2]
Leaky ReLU  : [-0.02 -0.01  0.    1.    2.  ]
ELU         : [-0.8647 -0.6321  0.      1.      2.    ]
Swish       : [-0.2384 -0.2689  0.      0.7311  1.7616]
Softplus    : [0.1269 0.3133 0.6931 1.3133 2.1269]
GELU        : [-0.0454 -0.1588  0.      0.8412  1.9546]
